In [1]:
import sys
from pathlib import Path

current_dir = Path.cwd()

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'bcosgnn').is_dir():
            return p
    raise RuntimeError('Could not locate repo root (pyproject.toml + bcosgnn/).')

project_root = find_repo_root(current_dir)

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Repo root added: {project_root}")

Repo root added: /Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn


In [12]:
import random
from collections import defaultdict
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.metrics import roc_auc_score
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_networkx
from torch_geometric.nn.conv import GINEConv
from torch_geometric.nn.aggr import MeanAggregation, SumAggregation

from tqdm.auto import tqdm

from bcos.modules import BcosLinear
from bcosgnn.explain_edge_attr import explain as explain_edge_attr
import os
import shutil
from torch_geometric.data import InMemoryDataset
import tqdm
import sys
import os
# Add the project root to the Python path
if project_root not in sys.path:
    sys.path.append(project_root)

import functools
import itertools
import operator
from typing import Any
import torch
from torch_geometric.data import Dataset, download_url
from torch.utils.data import random_split
import numpy as np
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from bcos.modules import BcosLinear, BcosSequential
from sklearn.model_selection import train_test_split
from torch.nn import BCEWithLogitsLoss
from torch_geometric.datasets import TUDataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing
from torch_geometric.nn.aggr import SumAggregation
from torch_geometric.utils import add_self_loops, degree
from torchmetrics import AUROC
from torchmetrics.classification import BinaryAccuracy
from tqdm import tqdm
import networkx as nx
import torch.nn.functional as F
from bcosgnn.explain import explain
from bcosgnn.evaluation import get_attribution_scores
import time

In [4]:
import importlib
import bcosgnn.evaluation
importlib.reload(bcosgnn.evaluation)
from bcosgnn.evaluation import get_attribution_scores

In [5]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [6]:
# Load Dataset
path = os.path.join(project_root, 'data', 'TUDataset')
dataset = TUDataset(root=path, name='MUTAG')

print(f'Dataset: {dataset}:')
print('====================')
print(f'Number of graphs: {len(dataset)}')
print(f'Number of features: {dataset.num_features}')
print(f'Number of classes: {dataset.num_classes}')

# Statistics
class_counts = {0: 0, 1: 0}
total_nodes = 0
class_nodes = {0: 0, 1: 0}

for data in dataset:
    y = data.y.item()
    class_counts[y] += 1
    total_nodes += data.num_nodes
    class_nodes[y] += data.num_nodes

print(f"\nClass Distribution:")
print(f"  Class 0 (Non-mutagenic): {class_counts[0]} graphs")
print(f"  Class 1 (Mutagenic):     {class_counts[1]} graphs")

print(f"\nNode Statistics:")
print(f"  Average nodes (Total):   {total_nodes / len(dataset):.2f}")
print(f"  Average nodes (Class 0): {class_nodes[0] / class_counts[0]:.2f}")
print(f"  Average nodes (Class 1): {class_nodes[1] / class_counts[1]:.2f}")

# Split Dataset
train_dataset, test_dataset = train_test_split(dataset, test_size=0.2, random_state=42, stratify=[d.y.item() for d in dataset])
print(f"\nTrain set size: {len(train_dataset)}")
print(f"Test set size: {len(test_dataset)}")

Dataset: MUTAG(188):
Number of graphs: 188
Number of features: 7
Number of classes: 2

Class Distribution:
  Class 0 (Non-mutagenic): 63 graphs
  Class 1 (Mutagenic):     125 graphs

Node Statistics:
  Average nodes (Total):   17.93
  Average nodes (Class 0): 13.94
  Average nodes (Class 1): 19.94

Train set size: 150
Test set size: 38


In [8]:
# -------------------------
# Cell 5: Model Classes
# -------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing, global_add_pool

# ==========================================
# 2. MODEL CLASSES
# ==========================================
class MaskableGINEConv(MessagePassing):
    def __init__(self, nn_mlp, train_eps=False):
        super().__init__(aggr='add') 
        self.nn = nn_mlp
        self.initial_eps = 0.0
        if train_eps:
            self.initial_eps = torch.nn.Parameter(torch.Tensor([0.0]))
            
    def forward(self, x, edge_index, edge_attr, edge_weight=None):
        out = self.propagate(edge_index, x=x, edge_attr=edge_attr, edge_weight=edge_weight)
        x_r = x[1] if isinstance(x, tuple) else x
        out = out + (1 + self.initial_eps) * x_r
        return self.nn(out)

    def message(self, x_j, edge_attr, edge_weight):
        msg = F.relu(x_j + edge_attr)
        if edge_weight is not None:
            msg = msg * edge_weight.view(-1, 1)
        
        return msg



In [9]:
class GINE_Network(nn.Module):
    def __init__(self, in_channels, edge_dim, hidden_channels, out_channels):
        super().__init__()
        self.node_lin = nn.Linear(in_channels, hidden_channels)
        self.edge_lin = nn.Linear(edge_dim, hidden_channels)
        
        mlp1 = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels), 
            nn.BatchNorm1d(hidden_channels), 
            nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels)
        )
        self.conv1 = MaskableGINEConv(mlp1, train_eps=True)
        
        mlp2 = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels), 
            nn.BatchNorm1d(hidden_channels), 
            nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels)
        )
        self.conv2 = MaskableGINEConv(mlp2, train_eps=True)
        self.lin = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, edge_attr, edge_weight=None, batch=None):
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
            
        x = self.node_lin(x.float())
        edge_attr = self.edge_lin(edge_attr.float())
        
        x = self.conv1(x, edge_index, edge_attr, edge_weight)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_attr, edge_weight)
        x = F.relu(x)
        
        x = global_add_pool(x, batch)
        return self.lin(x)

class GSAT(nn.Module):
    def __init__(self, backbone, in_channels, edge_attr_dim, hidden_channels, temperature=1.0):
        super(GSAT, self).__init__()
        self.backbone = backbone
        self.temperature = temperature

        self.att_mlp = nn.Sequential(
            nn.Linear(in_channels * 2 + edge_attr_dim, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, 1)
        )
        
        for m in self.att_mlp.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)

    def get_mask(self, x, edge_index, edge_attr, training=True):
        row, col = edge_index
        edge_rep = torch.cat([x[row], x[col], edge_attr], dim=-1)
        edge_logits = self.att_mlp(edge_rep).view(-1)

        if training:
            u = torch.rand_like(edge_logits)
            noise = torch.log(u + 1e-8) - torch.log(1 - u + 1e-8)
            stochastic_mask = torch.sigmoid((edge_logits + noise) / self.temperature)
        else:
            stochastic_mask = torch.sigmoid(edge_logits)
            
        return stochastic_mask, edge_logits

    def forward(self, data, training=True):
        x, edge_index, edge_attr, batch = data.x.float(), data.edge_index, data.edge_attr.float(), data.batch
        mask, mask_logits = self.get_mask(x, edge_index, edge_attr, training=training)
        pred_logits = self.backbone(x, edge_index, edge_attr, edge_weight=mask, batch=batch)
        return pred_logits, mask, mask_logits

In [10]:
# ==========================================
# 3. LOSS FUNCTION
# ==========================================
def gsat_loss(pred_logits, ground_truth_labels, mask_logits, r=0.7, pred_loss_coef=1.0, info_loss_coef=1.0):
    criterion = nn.CrossEntropyLoss()
    pred_loss = criterion(pred_logits, ground_truth_labels)
    
    mask_probs = torch.sigmoid(mask_logits)
    prior_target = torch.full_like(mask_probs, 1.0 - r)
    info_loss = F.binary_cross_entropy(mask_probs, prior_target, reduction='mean')
    
    loss = (pred_loss_coef * pred_loss) + (info_loss_coef * info_loss)
    return loss, pred_loss, info_loss

In [17]:
# Make sure to import f1_score at the top of your notebook if you haven't already:
from sklearn.metrics import f1_score, roc_auc_score
import time
import torch
import numpy as np
from torch_geometric.loader import DataLoader

# ==========================================
# 4. EXPERIMENT LOOP
# ==========================================
SEEDS = [0, 1, 2, 3, 4]
EPOCHS = 100
LR = 1e-3
HIDDEN_DIM = 64
R_PRIOR = 0.7
INFO_LOSS_COEF = 1.0

# ADDED: "f1" to the results dictionary
results = {"acc": [], "f1": [], "auc": [], "jaccard": [], "avg_epoch_time": []}

print(f"\nStarting 5-Fold Run on: {DEVICE}")
print("-" * 80)

for run_id, seed in enumerate(SEEDS):
    print(f"Run {run_id+1}/{len(SEEDS)} (Seed: {seed})")
    
    set_seed(seed)
    dataset_shuffled = dataset.shuffle()
    
    train_size = int(len(dataset_shuffled) * 0.8)
    train_dataset = dataset_shuffled[:train_size]
    test_dataset = dataset_shuffled[train_size:]
    
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    backbone = GINE_Network(
        in_channels=dataset.num_features,
        edge_dim=dataset.num_edge_features,
        hidden_channels=HIDDEN_DIM,
        out_channels=dataset.num_classes   
    )
    
    model = GSAT(
        backbone=backbone,
        in_channels=dataset.num_features,
        edge_attr_dim=dataset.num_edge_features,
        hidden_channels=HIDDEN_DIM,
        temperature=1.0
    ).to(DEVICE)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    seed_epoch_times = []
    
    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.time()
        model.train()
        model.temperature = 1.0 - (epoch / EPOCHS) * (1.0 - 0.1)
        
        for data in train_loader:
            data = data.to(DEVICE)
            optimizer.zero_grad()
            logits, mask, mask_logits = model(data, training=True)
            loss, _, _ = gsat_loss(logits, data.y, mask_logits, r=R_PRIOR, pred_loss_coef=1.0, info_loss_coef=INFO_LOSS_COEF)
            loss.backward()
            optimizer.step()
            
        seed_epoch_times.append(time.time() - epoch_start)

    avg_time = np.mean(seed_epoch_times)
    results["avg_epoch_time"].append(avg_time)
            
    # Evaluation
    model.eval()
    correct_graphs, total_graphs = 0, 0
    run_jaccards, all_gt, all_pred = [], [], []
    
    # ADDED: Lists to store true labels and predictions for F1
    run_true_labels = []
    run_pred_labels = []
    
    has_explanations = False
    
    with torch.no_grad():
        for data in test_loader:
            data = data.to(DEVICE)
            logits, mask, _ = model(data, training=False)
            
            # Accuracy and F1 Prep
            pred_classes = logits.argmax(dim=1)
            correct_graphs += (pred_classes == data.y).sum().item()
            total_graphs += data.num_graphs
            
            # ADDED: Store labels for F1 calculation
            run_true_labels.extend(data.y.cpu().numpy())
            run_pred_labels.extend(pred_classes.cpu().numpy())
            
            # Explanation Metrics (AUC/Jaccard)
            if hasattr(data, 'explanation_mask') and data.explanation_mask is not None:
                has_explanations = True
                row, col = data.edge_index
                node_mask = data.explanation_mask.bool()
                edge_gt = (node_mask[row] & node_mask[col]).cpu().numpy()
                all_gt.extend(edge_gt)
                all_pred.extend(mask.cpu().numpy())
                
                edge_batch = data.batch[row]
                pred_binary = (mask > 0.5).float()
                gt_binary = (node_mask[row] & node_mask[col]).float()
                
                intersection = pred_binary * gt_binary
                union = (pred_binary + gt_binary).clamp(max=1.0)
                
                graph_intersection = torch.zeros(data.num_graphs, device=DEVICE).scatter_add_(0, edge_batch, intersection)
                graph_union = torch.zeros(data.num_graphs, device=DEVICE).scatter_add_(0, edge_batch, union)
                
                run_jaccards.extend((graph_intersection / (graph_union + 1e-8)).cpu().tolist())

    final_acc = correct_graphs / total_graphs
    
    # ADDED: Calculate F1 Score using 'macro' average to handle imbalance
    final_f1 = f1_score(run_true_labels, run_pred_labels, average='macro')
    
    final_auc = roc_auc_score(all_gt, all_pred) if has_explanations and len(set(all_gt)) > 1 else np.nan
    final_jaccard = np.mean(run_jaccards) if has_explanations else np.nan
    
    # ADDED: Print F1 score
    print(f"  -> Avg Epoch Time: {avg_time:.4f}s | Acc: {final_acc:.4f} | F1: {final_f1:.4f}", end="")
    if has_explanations:
        print(f" | AUC: {final_auc:.4f} | Jaccard: {final_jaccard:.4f}")
    else:
        print(" | (No explanation ground truths in dataset)")
        
    results["acc"].append(final_acc)
    results["f1"].append(final_f1)   # ADDED: Save to results
    if has_explanations:
        results["auc"].append(final_auc)
        results["jaccard"].append(final_jaccard)

print("-" * 80)
print(f"Final Results over {len(SEEDS)} Seeds:")
print(f"Avg Time Per Epoch: {np.mean(results['avg_epoch_time']):.4f}s ± {np.std(results['avg_epoch_time']):.4f}")
print(f"Test Accuracy:      {np.mean(results['acc']):.4f} ± {np.std(results['acc']):.4f}")
print(f"Test F1 Score:      {np.mean(results['f1']):.4f} ± {np.std(results['f1']):.4f}") # ADDED: Final F1 summary

if len(results["auc"]) > 0:
    print(f"Explanation AUC:    {np.mean(results['auc']):.4f} ± {np.std(results['auc']):.4f}")
    print(f"Explanation Jaccard:{np.mean(results['jaccard']):.4f} ± {np.std(results['jaccard']):.4f}")


Starting 5-Fold Run on: cpu
--------------------------------------------------------------------------------
Run 1/5 (Seed: 0)
  -> Avg Epoch Time: 0.0193s | Acc: 0.7632 | F1: 0.6842 | (No explanation ground truths in dataset)
Run 2/5 (Seed: 1)
  -> Avg Epoch Time: 0.0193s | Acc: 0.7632 | F1: 0.6842 | (No explanation ground truths in dataset)
Run 2/5 (Seed: 1)
  -> Avg Epoch Time: 0.0179s | Acc: 0.7105 | F1: 0.6571 | (No explanation ground truths in dataset)
Run 3/5 (Seed: 2)
  -> Avg Epoch Time: 0.0179s | Acc: 0.7105 | F1: 0.6571 | (No explanation ground truths in dataset)
Run 3/5 (Seed: 2)
  -> Avg Epoch Time: 0.0182s | Acc: 0.6842 | F1: 0.5632 | (No explanation ground truths in dataset)
Run 4/5 (Seed: 3)
  -> Avg Epoch Time: 0.0182s | Acc: 0.6842 | F1: 0.5632 | (No explanation ground truths in dataset)
Run 4/5 (Seed: 3)
  -> Avg Epoch Time: 0.0179s | Acc: 0.6842 | F1: 0.6162 | (No explanation ground truths in dataset)
Run 5/5 (Seed: 4)
  -> Avg Epoch Time: 0.0179s | Acc: 0.6842 | F